# FIFA 19 Player Analysis & Valuation Project
**Author:** [Your Name]  
**Date:** 2025  
**Tools:** Python, Pandas, Scikit-Learn, Tableau

## 📌 Project Overview
This project performs a comprehensive **Exploratory Data Analysis (EDA)** and **Data Cleaning** pipeline on the FIFA 19 player dataset. The goal is to prepare a pristine dataset for dashboarding in Tableau, with a specific focus on handling missing financial data using Machine Learning.

### 🔑 Key Features
* **Advanced Imputation:** Used Random Forest Regressors to predict missing `Release Clause` values based on player market value and wages.
* **Feature Engineering:** Created new metrics like `Contract_Years_Remaining` and regional mappings.
* **Data Transformation:** Parsed complex string data (currency, height, weights, equations) into usable numerical formats.

## 1️⃣ Phase 1: Environment Setup & Data Initialization
**Objective:** Import necessary libraries and load the raw dataset.
* **Libraries:** `pandas`, `numpy`, `matplotlib`.
* **Input:** `fifa19.csv`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")


In [2]:
df = pd.read_csv("D:/desktop/MY_DOC/Projects/Comprehensive Analysis FIFA 19/RawData/fifa19.csv")
df.head(5)


,Unnamed: 0,ID,Name,Age,Photo,Nationality,Flag,Overall,Potential,Club,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
0,0,158023,L. Messi,31,https://cdn.sofifa.org/players/4/19/158023.png,Argentina,https://cdn.sofifa.org/flags/52.png,94,94,FC Barcelona,...,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0,€226.5M
1,1,20801,Cristiano Ronaldo,33,https://cdn.sofifa.org/players/4/19/20801.png,Portugal,https://cdn.sofifa.org/flags/38.png,94,94,Juventus,...,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0,€127.1M
2,2,190871,Neymar Jr,26,https://cdn.sofifa.org/players/4/19/190871.png,Brazil,https://cdn.sofifa.org/flags/54.png,92,93,Paris Saint-Germain,...,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0,€228.1M
3,3,193080,De Gea,27,https://cdn.sofifa.org/players/4/19/193080.png,Spain,https://cdn.sofifa.org/flags/45.png,91,93,Manchester United,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
4,4,192985,K. De Bruyne,27,https://cdn.sofifa.org/players/4/19/192985.png,Belgium,https://cdn.sofifa.org/flags/7.png,91,92,Manchester City,...,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0,€196.4M


In [3]:
print("Dataset Shape:", df.shape)


Dataset Shape: (18207, 89)


In [4]:
len(df['ID'].unique())

18207

## 2️⃣ Phase 2: Initial Data Cleaning
**Objective:** Remove redundancy and format the dataframe structure.
* **Dropped Columns:** Removed non-analytical artifacts (`Photo`, `Flag`, `Club Logo`) and loan-specific tracking (`Loaned From`).
* **Sanity Checks:** Verified shape and identified initial null patterns.

In [5]:
df.drop(columns=["Unnamed: 0", "Photo", "Flag", "Club Logo"], inplace=True)


In [6]:
df=df.drop_duplicates()

In [7]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Loaned From,16943,93.06
1,LM,2085,11.45
2,RAM,2085,11.45
3,CAM,2085,11.45
4,CF,2085,11.45
5,RF,2085,11.45
6,RW,2085,11.45
7,LF,2085,11.45
8,ST,2085,11.45
9,RS,2085,11.45


In [8]:
df = df.drop(columns=["Loaned From"])


## 3️⃣ Phase 3: Domain-Specific Logic Handling
**Objective:** Correct structural missing data for Goalkeepers.
* **Context:** Goalkeepers naturally lack outfield ratings (e.g., `Crossing`, `Finishing`). In the raw data, these appear as `NaN`.
* **Action:** Imputed these specific `NaN` values with `0` to distinguish them from genuinely missing data.

In [9]:
df[df['LM'].isnull()].head(5)

,ID,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Special,...,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause
3,193080,De Gea,27,Spain,91,93,Manchester United,€72M,€260K,1471,...,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0,€138.6M
9,200389,J. Oblak,25,Slovenia,90,93,Atlético Madrid,€68M,€94K,1331,...,70.0,27.0,12.0,18.0,86.0,92.0,78.0,88.0,89.0,€144.5M
18,192448,M. ter Stegen,26,Germany,89,92,FC Barcelona,€58M,€240K,1328,...,69.0,25.0,13.0,10.0,87.0,85.0,88.0,85.0,90.0,€123.3M
19,192119,T. Courtois,26,Belgium,89,90,Real Madrid,€53.5M,€240K,1311,...,66.0,20.0,18.0,16.0,85.0,91.0,72.0,86.0,88.0,€113.7M
22,167495,M. Neuer,32,Germany,89,89,FC Bayern München,€38M,€130K,1473,...,70.0,17.0,10.0,11.0,90.0,86.0,91.0,87.0,87.0,€62.7M


In [10]:
null_rep_df = df[df["LM"].isnull()]
null_rep_gk = null_rep_df[null_rep_df["Position"] == "GK"]
len(null_rep_gk)


2025

In [11]:
gk_row = df[df["Position"] == "GK"].iloc[0]
gk_row.to_frame()

,3
ID,193080
Name,De Gea
Age,27
Nationality,Spain
Overall,91
Potential,93
Club,Manchester United
Value,€72M
Wage,€260K
Special,1471


In [12]:
outfield_position_cols = [
    "LS", "ST", "RS",
    "LW", "LF", "CF", "RF", "RW",
    "LAM", "CAM", "RAM",
    "LM", "LCM", "CM", "RCM", "RM",
    "LWB", "LDM", "CDM", "RDM", "RWB",
    "LB", "LCB", "CB", "RCB", "RB"
]


df.loc[
    df["Position"] == "GK",
    outfield_position_cols
] = df.loc[
    df["Position"] == "GK",
    outfield_position_cols
].fillna(0)



C:\Users\terbo\AppData\Local\Temp\ipykernel_296\376399933.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(0)


In [13]:
df.loc[df["Position"] == "GK", outfield_position_cols].isnull().sum()


LS     0
ST     0
RS     0
LW     0
LF     0
CF     0
RF     0
RW     0
LAM    0
CAM    0
RAM    0
LM     0
LCM    0
CM     0
RCM    0
RM     0
LWB    0
LDM    0
CDM    0
RDM    0
RWB    0
LB     0
LCB    0
CB     0
RCB    0
RB     0
dtype: int64

In [14]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Release Clause,1564,8.59
1,Joined,1553,8.53
2,Contract Valid Until,289,1.59
3,Club,241,1.32
4,RF,60,0.33
5,Jersey Number,60,0.33
6,LW,60,0.33
7,LS,60,0.33
8,RS,60,0.33
9,LF,60,0.33


## 4️⃣ Phase 4: Financial Data Parsing
**Objective:** Convert string-formatted currency fields into computational floats.
* **Transformation:** Parsed `Value`, `Wage`, and `Release Clause` (e.g., converted "€110.5M" $\rightarrow$ `110500000.0`).
* **Edge Case:** Identified "Free Agents" (Zero Value/Wage) to prevent skewing distribution metrics.

In [15]:
def convert_currency(value):
    if isinstance(value, str):
        value = value.replace('€', '').replace('M', 'e6').replace('K', 'e3')
        return float(eval(value))
    return value

money_cols = ["Value", "Wage", "Release Clause"]

for col in money_cols:
    df[col] = df[col].apply(convert_currency)
df[["Value", "Wage", "Release Clause"]].head(5)


,Value,Wage,Release Clause
0,110500000.0,565000.0,226500000.0
1,77000000.0,405000.0,127100000.0
2,118500000.0,290000.0,228100000.0
3,72000000.0,260000.0,138600000.0
4,102000000.0,355000.0,196400000.0


In [16]:
null_position = df[df["Position"].isnull()].iloc[0]
null_position.to_frame()

,5018
ID,153160
Name,R. Raldes
Age,37
Nationality,Bolivia
Overall,70
Potential,70
Club,NaN
Value,0.0
Wage,0.0
Special,1574


In [17]:
null_cols = [
    "Position",
    "Club",
    "Joined",
    "Contract Valid Until"
]

money_cols = ["Value", "Wage"]
filtered_df = df[
    df[null_cols].isnull().all(axis=1) &
    (df[money_cols] == 0.0).all(axis=1)
]
len(filtered_df)


12

In [18]:
df = df.drop(filtered_df.index)
df.reset_index(drop=True, inplace=True)


In [19]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Release Clause,1552,8.53
1,Joined,1541,8.47
2,Contract Valid Until,277,1.52
3,Club,229,1.26
4,Weak Foot,48,0.26
5,International Reputation,48,0.26
6,Skill Moves,48,0.26
7,Body Type,48,0.26
8,Real Face,48,0.26
9,Work Rate,48,0.26


In [20]:
df = df.dropna(subset=["International Reputation"])
df.reset_index(drop=True, inplace=True)


In [21]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Release Clause,1504,8.29
1,Joined,1493,8.23
2,Club,229,1.26
3,Contract Valid Until,229,1.26


In [22]:
rc_null_df = df[df["Release Clause"].isnull()]
pd.DataFrame({
    "Zero Value & Wage Count": [((rc_null_df["Value"] == 0) & (rc_null_df["Wage"] == 0)).sum()],
    "Total Players": [len(rc_null_df)]
})


,Zero Value & Wage Count,Total Players
0,229,1504


In [23]:
zero_value_wage_mask = (
    (rc_null_df["Value"] == 0) &
    (rc_null_df["Wage"] == 0)
)
pd.DataFrame({
    "Zero Value & Wage Players": [zero_value_wage_mask.sum()],
    "Also Null Contract": [
        rc_null_df.loc[zero_value_wage_mask, "Contract Valid Until"].isnull().sum()
    ]
})


,Zero Value & Wage Players,Also Null Contract
0,229,229


In [24]:
zero_value_wage_contract_null = rc_null_df[
    zero_value_wage_mask &
    rc_null_df["Contract Valid Until"].isnull()
]
len(zero_value_wage_contract_null)

229

In [25]:
df.loc[
    (df["Value"] == 0) &
    (df["Wage"] == 0) &
    (df["Contract Valid Until"].isnull()),
    "Club"
] = "Free Agent"


In [26]:
df.loc[df["Club"] == "Free Agent", "Release Clause"] = 0


In [27]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Joined,1493,8.23
1,Release Clause,1275,7.03
2,Contract Valid Until,229,1.26


## 5️⃣ Phase 5: Time-Series & Contract Analysis
**Objective:** Standardize dates to calculate player stability and contract risks.
* **Date Parsing:** Converted `Joined` and `Contract Valid Until` to Datetime objects.
* **Imputation:** Handled missing contract expiration dates by adding a 3-year offset to the `Joined` date.
    * *Rationale:* Exploratory analysis of the dataset distribution revealed that **3 years** is the modal (most frequent) contract length, making it the statistically safest imputation value.
* **Feature Engineering:**
    * `Years_At_Club`: Measures loyalty.
    * `Contract_Years_Remaining`: Critical for transfer value analysis.

In [28]:
df['Contract Valid Until'].unique()

array(['2021', '2022', '2020', '2023', '2019', '2024', 'Jun 30, 2019',
       '2025', '2026', 'Dec 31, 2018', '2018', nan, 'May 31, 2020',
       'Jun 30, 2020', 'May 31, 2019', 'Dec 31, 2019', 'Jan 1, 2019',
       'Jun 1, 2019', 'Jan 4, 2019', 'Jan 31, 2019', 'Jan 7, 2019',
       'Jan 2, 2019', 'Jan 6, 2019', 'Oct 14, 2019', 'Jan 3, 2019',
       'May 4, 2019', 'Jan 12, 2019', 'Jan 25, 2019', 'Jan 18, 2019',
       'Dec 1, 2019', 'Nov 30, 2018', 'Feb 27, 2020', 'Jan 5, 2019',
       'Jan 15, 2019', 'Jan 30, 2019', 'Jan 11, 2019', 'Jan 20, 2019'],
      dtype=object)

In [29]:
df['Joined'].unique()

array(['Jul 1, 2004', 'Jul 10, 2018', 'Aug 3, 2017', ..., 'May 22, 2017',
       'Nov 6, 2016', 'Nov 27, 2018'], shape=(1737,), dtype=object)

In [30]:
def parse_contract_date(x):
    if pd.isna(x):
        return pd.NaT
    x = str(x).strip()
    
    # Case 1: Year only (e.g. "2022")
    if x.isdigit() and len(x) == 4:
        return pd.to_datetime(f"{x}-12-31")
    
    # Case 2: Full date string
    return pd.to_datetime(x, errors="coerce")


In [31]:
df["Contract Valid Until"] = df["Contract Valid Until"].apply(parse_contract_date)
df["Contract Valid Until"].dtype


dtype('<M8[ns]')

In [32]:
df["Contract Valid Until"].isnull().sum()


np.int64(229)

In [33]:
df["Joined"] = pd.to_datetime(df["Joined"], errors="coerce")


In [34]:
complete_contracts = df[
    df["Joined"].notnull() &
    df["Contract Valid Until"].notnull()
].copy()
len(complete_contracts)


16654

In [35]:
complete_contracts["Contract_Duration_Years"] = (
    complete_contracts["Contract Valid Until"].dt.year -
    complete_contracts["Joined"].dt.year
)



In [36]:
Q1 = complete_contracts["Contract_Duration_Years"].quantile(0.25)
Q3 = complete_contracts["Contract_Duration_Years"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

lower_bound, upper_bound

(np.float64(-2.5), np.float64(9.5))

In [37]:
complete_contracts = complete_contracts[
    (complete_contracts["Contract_Duration_Years"] > 0) &
    (complete_contracts["Contract_Duration_Years"] <= 10)
]

In [38]:
complete_contracts["Contract_Duration_Years"].describe()


count    16075.000000
mean         3.663390
std          1.939639
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max         10.000000
Name: Contract_Duration_Years, dtype: float64

In [39]:
df["Joined"] = pd.to_datetime(df["Joined"], errors="coerce")
df["Contract Valid Until"] = pd.to_datetime(
    df["Contract Valid Until"], errors="coerce"
)


In [40]:
mask_add_end = (
    df["Joined"].notnull() &
    df["Contract Valid Until"].isnull()
)

df.loc[mask_add_end, "Contract Valid Until"] = (
    df.loc[mask_add_end, "Joined"] + pd.DateOffset(years=3)
)


In [41]:
mask_sub_joined = (
    df["Joined"].isnull() &
    df["Contract Valid Until"].notnull()
)

df.loc[mask_sub_joined, "Joined"] = (
    df.loc[mask_sub_joined, "Contract Valid Until"] - pd.DateOffset(years=3)
)


In [42]:
pd.set_option("display.max_rows", None)

null_df = (
    df.isnull()
      .sum()
      .loc[lambda x: x > 0]
      .sort_values(ascending=False)
      .reset_index()
)

null_df.columns = ["Column Name", "Null Count"]
null_df["Null Percentage (%)"] = (null_df["Null Count"] / len(df) * 100).round(2)

null_df


,Column Name,Null Count,Null Percentage (%)
0,Release Clause,1275,7.03
1,Joined,229,1.26
2,Contract Valid Until,229,1.26


In [43]:
mask_free_agents = (
    df["Joined"].isnull() &
    df["Contract Valid Until"].isnull() &
    df["Release Clause"].isnull()
)
mask_free_agents.sum()


np.int64(0)

In [44]:
check_null_dates = (
    df["Joined"].isnull() &
    df["Contract Valid Until"].isnull()
)
check_null_dates.sum()

np.int64(229)

In [45]:
df.loc[check_null_dates, ["Name", "Club"]].head(5)


,Name,Club
452,L. Paredes,Free Agent
538,A. Granqvist,Free Agent
568,A. Lunev,Free Agent
677,I. Smolnikov,Free Agent
874,A. Dzyuba,Free Agent


In [46]:
df.loc[check_null_dates, "Club"].value_counts(dropna=False)


Club
Free Agent    229
Name: count, dtype: int64

In [47]:
reference_year = 2019
df['Years_At_Club'] = reference_year - df['Joined'].dt.year
df['Contract_Years_Remaining'] = df['Contract Valid Until'].dt.year - reference_year

In [48]:
df['Contract_Years_Remaining'] = df['Contract_Years_Remaining'].clip(lower=0)


In [49]:
df['Contract_Info_Missing'] = (
    df['Years_At_Club'].isna() & df['Contract_Years_Remaining'].isna()
).astype(int)


df['Years_At_Club'].fillna(df['Years_At_Club'].median(), inplace=True)
df['Contract_Years_Remaining'].fillna(
    df['Contract_Years_Remaining'].median(),
    inplace=True
)


df.drop(columns=['Joined', 'Contract Valid Until'], inplace=True)


C:\Users\terbo\AppData\Local\Temp\ipykernel_296\2068804957.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Years_At_Club'].fillna(df['Years_At_Club'].median(), inplace=True)
C:\Users\terbo\AppData\Local\Temp\ipykernel_296\2068804957.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

In [50]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18147 entries, 0 to 18146
Data columns (total 85 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        18147 non-null  int64  
 1   Name                      18147 non-null  object 
 2   Age                       18147 non-null  int64  
 3   Nationality               18147 non-null  object 
 4   Overall                   18147 non-null  int64  
 5   Potential                 18147 non-null  int64  
 6   Club                      18147 non-null  object 
 7   Value                     18147 non-null  float64
 8   Wage                      18147 non-null  float64
 9   Special                   18147 non-null  int64  
 10  Preferred Foot            18147 non-null  object 
 11  International Reputation  18147 non-null  float64
 12  Weak Foot                 18147 non-null  float64
 13  Skill Moves               18147 non-null  float64
 14  Work R

## 6️⃣ Phase 6: Physical & Skill Standardization
**Objective:** Normalize physical units and evaluate composite player ratings.
* **Unit Conversion:**
    * `Height`: Feet/Inches $\rightarrow$ Centimeters.
    * `Weight`: Lbs $\rightarrow$ Kilograms.
* **Encoding:** Simplified categorical variables (`Work Rate`, `Body Type`) for correlation analysis.
* **Skill Parsing:** Evaluated mathematical strings in rating columns (e.g., "88+2" $\rightarrow$ `90`).


In [51]:
def height_to_cm(height):
    if pd.isna(height):
        return np.nan
    try:
        feet, inches = height.split("'")
        inches = inches.replace('"', '')
        return round(int(feet) * 30.48 + int(inches) * 2.54, 2)
    except:
        return np.nan

df['Height'] = df['Height'].apply(height_to_cm)

In [52]:
def weight_to_kg(weight):
    if pd.isna(weight):
        return np.nan
    try:
        return round(float(weight.replace('lbs', '')) * 0.453592, 2)
    except:
        return np.nan

df['Weight'] = df['Weight'].apply(weight_to_kg)


In [53]:
df['Preferred Foot'].unique()

array(['Left', 'Right'], dtype=object)

In [54]:
df['Preferred Foot'] = df['Preferred Foot'].map({'Left': 0, 'Right': 1})


In [55]:
df['Work Rate'].unique()

array(['Medium/ Medium', 'High/ Low', 'High/ Medium', 'High/ High',
       'Medium/ High', 'Medium/ Low', 'Low/ High', 'Low/ Medium',
       'Low/ Low'], dtype=object)

In [56]:
df['Attack Work Rate']=df['Work Rate'].apply(lambda x: x.split('/ ')[0])
df['Defence Work Rate']=df['Work Rate'].apply(lambda x: x.split('/ ')[1])
W_Dec={'High':2,'Medium':1,'Low':0}
df['Attack Work Rate']=df['Attack Work Rate'].map(W_Dec)
df['Defence Work Rate']=df['Defence Work Rate'].map(W_Dec)
df = df.drop(columns=["Work Rate"])
df.head(5)

,ID,Name,Age,Nationality,Overall,Potential,Club,Value,Wage,Special,...,GKHandling,GKKicking,GKPositioning,GKReflexes,Release Clause,Years_At_Club,Contract_Years_Remaining,Contract_Info_Missing,Attack Work Rate,Defence Work Rate
0,158023,L. Messi,31,Argentina,94,94,FC Barcelona,110500000.0,565000.0,2202,...,11.0,15.0,14.0,8.0,226500000.0,15.0,2.0,0,1,1
1,20801,Cristiano Ronaldo,33,Portugal,94,94,Juventus,77000000.0,405000.0,2228,...,11.0,15.0,14.0,11.0,127100000.0,1.0,3.0,0,2,0
2,190871,Neymar Jr,26,Brazil,92,93,Paris Saint-Germain,118500000.0,290000.0,2143,...,9.0,15.0,15.0,11.0,228100000.0,2.0,3.0,0,2,1
3,193080,De Gea,27,Spain,91,93,Manchester United,72000000.0,260000.0,1471,...,85.0,87.0,88.0,94.0,138600000.0,8.0,1.0,0,1,1
4,192985,K. De Bruyne,27,Belgium,91,92,Manchester City,102000000.0,355000.0,2281,...,13.0,5.0,10.0,13.0,196400000.0,4.0,4.0,0,2,2


In [57]:
body_type_counts = df['Body Type'].value_counts()
body_type_counts

Body Type
Normal                 10588
Lean                    6412
Stocky                  1140
Messi                      1
C. Ronaldo                 1
Neymar                     1
Courtois                   1
PLAYER_BODY_TYPE_25        1
Shaqiri                    1
Akinfenwa                  1
Name: count, dtype: int64

In [58]:
player_name = df.loc[df['Body Type'] == 'PLAYER_BODY_TYPE_25', 'Name']
print(player_name)

26    M. Salah
Name: Name, dtype: object


In [59]:
body_type_mapping = {
    'Messi': 'Stocky',
    'Neymar': 'Lean',
    'Shaqiri': 'Stocky',
    'Akinfenwa': 'Stocky',
    'C. Ronaldo': 'Lean',
    'Courtois': 'Lean',
    'PLAYER_BODY_TYPE_25': 'Lean'
}

df['Body Type'] = df['Body Type'].replace(body_type_mapping)

In [60]:
df['Body Type'].unique()

array(['Stocky', 'Lean', 'Normal'], dtype=object)

In [61]:
df['Body Type'] = df['Body Type'].map({
    'Lean': 0,
    'Normal': 1,
    'Stocky': 2
})


In [62]:
df['Real Face'].unique()

array(['Yes', 'No'], dtype=object)

In [63]:
df['Real Face'] = df['Real Face'].map({'No': 0, 'Yes': 1})


In [64]:
df['Position'].unique()

array(['RF', 'ST', 'LW', 'GK', 'RCM', 'LF', 'RS', 'RCB', 'LCM', 'CB',
       'LDM', 'CAM', 'CDM', 'LS', 'LCB', 'RM', 'LAM', 'LM', 'LB', 'RDM',
       'RW', 'CM', 'RB', 'RAM', 'CF', 'RWB', 'LWB'], dtype=object)

In [65]:
object_cols = df.select_dtypes(include='object').columns.tolist()
object_cols

['Name',
 'Nationality',
 'Club',
 'Position',
 'LS',
 'ST',
 'RS',
 'LW',
 'LF',
 'CF',
 'RF',
 'RW',
 'LAM',
 'CAM',
 'RAM',
 'LM',
 'LCM',
 'CM',
 'RCM',
 'RM',
 'LWB',
 'LDM',
 'CDM',
 'RDM',
 'RWB',
 'LB',
 'LCB',
 'CB',
 'RCB',
 'RB']

In [66]:
df['RM'].unique()

array(['91+2', '88+3', 0, '89+3', '86+3', '84+5', '72+3', '81+3', '82+3',
       '63+3', '85+3', '79+3', '76+3', '83+3', '56+3', '71+3', '70+3',
       '78+3', '69+3', '74+3', '55+3', '77+3', '66+3', '83+2', '84+2',
       '67+3', '84+3', '64+3', '80+3', '65+3', '68+3', '60+3', '60+2',
       '80+2', '53+3', '61+3', '78+2', '75+3', '62+3', '58+3', '76+4',
       '62+2', '53+2', '64+2', '82+2', '52+2', '79+2', '81+2', '77+2',
       '57+3', '61+2', '71+2', '66+2', '57+2', '74+2', '54+2', '68+2',
       '59+2', '72+2', '48+3', '65+2', '75+2', '76+2', '67+2', '56+2',
       '58+2', '63+2', '73+2', '70+2', '54+3', '59+3', '55+2', '69+2',
       '73+3', '51+3', '51+2', '48+2', '47+2', '50+2', '44+3', '45+2',
       '49+2', '44+2', '46+2', '39+2', '43+2', '41+2', '38+2', '42+2',
       '40+2', '37+2', '36+2', '29+2', '35+2', '34+2', '33+2', '32+2',
       '31+2', '30+2', '28+2', '27+2'], dtype=object)

In [67]:
position_rating_cols = ['LS','ST','RS','LW','LF','CF','RF','RW','LAM','CAM','RAM','LM','LCM','CM','RCM',
 'RM','LWB','LDM','CDM','RDM','RWB','LB','LCB','CB','RCB','RB']

def parse_position_rating(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return val
    if isinstance(val, str):
        if '+' in val:
            base, boost = val.split('+')
            return float(base) + float(boost)
        else:
            return float(val)
    return np.nan
df[position_rating_cols] = df[position_rating_cols].applymap(parse_position_rating)


C:\Users\terbo\AppData\Local\Temp\ipykernel_296\37564800.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[position_rating_cols] = df[position_rating_cols].applymap(parse_position_rating)


In [68]:
int_cols = df.select_dtypes(include='int').columns.tolist()
float_cols = df.select_dtypes(include='float').columns.tolist()
object_cols = df.select_dtypes(include='object').columns.tolist()

print("🔢 Integer columns:")
print(int_cols)

print("\n📈 Float columns:")
print(float_cols)

print("\n📝 Object columns:")
print(object_cols)


🔢 Integer columns:
['ID', 'Age', 'Overall', 'Potential', 'Special', 'Preferred Foot', 'Body Type', 'Real Face', 'Contract_Info_Missing', 'Attack Work Rate', 'Defence Work Rate']

📈 Float columns:
['Value', 'Wage', 'International Reputation', 'Weak Foot', 'Skill Moves', 'Jersey Number', 'Height', 'Weight', 'LS', 'ST', 'RS', 'LW', 'LF', 'CF', 'RF', 'RW', 'LAM', 'CAM', 'RAM', 'LM', 'LCM', 'CM', 'RCM', 'RM', 'LWB', 'LDM', 'CDM', 'RDM', 'RWB', 'LB', 'LCB', 'CB', 'RCB', 'RB', 'Crossing', 'Finishing', 'HeadingAccuracy', 'ShortPassing', 'Volleys', 'Dribbling', 'Curve', 'FKAccuracy', 'LongPassing', 'BallControl', 'Acceleration', 'SprintSpeed', 'Agility', 'Reactions', 'Balance', 'ShotPower', 'Jumping', 'Stamina', 'Strength', 'LongShots', 'Aggression', 'Interceptions', 'Positioning', 'Vision', 'Penalties', 'Composure', 'Marking', 'StandingTackle', 'SlidingTackle', 'GKDiving', 'GKHandling', 'GKKicking', 'GKPositioning', 'GKReflexes', 'Release Clause', 'Years_At_Club', 'Contract_Years_Remaining']



## 7️⃣ Phase 7: Geographic Feature Engineering
**Objective:** Aggregate high-cardinality `Nationality` data into broader regions.
* **Mapping:** Grouped ~160 nationalities into macro-regions (Europe, South America, Asia, etc.).
* **Purpose:** Enables regional market analysis in the final dashboard.

In [69]:
Europe = {
    'Albania','Andorra','Armenia','Austria','Azerbaijan','Belarus','Belgium',
    'Bosnia Herzegovina','Bulgaria','Croatia','Cyprus','Czech Republic',
    'Denmark','England','Estonia','Finland','France','FYR Macedonia',
    'Faroe Islands','Georgia','Germany','Greece','Hungary','Iceland',
    'Ireland','Republic of Ireland','Italy','Kosovo','Latvia','Liechtenstein',
    'Lithuania','Luxembourg','Malta','Moldova','Montenegro','Netherlands',
    'Northern Ireland','Norway','Poland','Portugal','Romania','Russia',
    'Scotland','Serbia','Slovakia','Slovenia','Spain','Sweden','Switzerland',
    'Turkey','Ukraine','Wales'
}

South_America = {
    'Argentina','Bolivia','Brazil','Chile','Colombia','Ecuador','Paraguay',
    'Peru','Uruguay','Venezuela'
}

North_America = {
    'Antigua & Barbuda','Barbados','Belize','Bermuda','Canada',
    'Costa Rica','Cuba','Curacao','Dominican Republic','El Salvador',
    'Grenada','Guatemala','Guyana','Haiti','Honduras','Jamaica','Mexico',
    'Montserrat','Nicaragua','Panama','Puerto Rico','St Kitts Nevis',
    'St Lucia','Suriname','Trinidad & Tobago','United States'
}

Africa = {
    'Algeria','Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Cape Verde','Central African Rep.','Chad','Comoros',
    'Congo','DR Congo','Egypt','Equatorial Guinea','Eritrea','Ethiopia',
    'Gabon','Gambia','Ghana','Guinea','Guinea Bissau','Ivory Coast',
    'Kenya','Liberia','Libya','Madagascar','Mali','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria','Rwanda',
    'Senegal','Sierra Leone','South Africa','South Sudan','Sudan',
    'São Tomé & Príncipe','Tanzania','Togo','Tunisia','Uganda','Zambia',
    'Zimbabwe'
}

Asia = {
    'Afghanistan','China PR','Hong Kong','India','Indonesia','Iran','Iraq',
    'Israel','Japan','Jordan','Kazakhstan','Korea DPR','Korea Republic',
    'Kuwait','Lebanon','Oman','Palestine','Philippines','Qatar',
    'Saudi Arabia','Syria','Thailand','United Arab Emirates','Uzbekistan'
}

Oceania = {
    'Australia','Fiji','Guam','New Caledonia','New Zealand'
}


In [70]:
def map_nationality_to_region(country):
    if country in Europe:
        return 'Europe'
    elif country in South_America:
        return 'South_America'
    elif country in North_America:
        return 'North_America'
    elif country in Africa:
        return 'Africa'
    elif country in Asia:
        return 'Asia'
    elif country in Oceania:
        return 'Oceania'
    else:
        return 'OTHER'  
    
df['Nationality_Region'] = df['Nationality'].apply(map_nationality_to_region)
df.drop(columns=['Nationality'], inplace=True)
df['Nationality_Region'].value_counts()



Nationality_Region
Europe           10896
South_America     3169
Asia              1639
Africa            1215
North_America      945
Oceania            283
Name: count, dtype: int64

In [71]:
df_display= df.copy()
processed_df=df.drop(columns=['Name', 'Club', 'ID', 'Position'], inplace=True)


In [72]:
processed_df = pd.get_dummies(df, columns=['Nationality_Region'], prefix='Region')
reg_cols = [c for c in processed_df.columns if c.startswith('Region_')]
processed_df[reg_cols] = processed_df[reg_cols].astype(float)
processed_df.head(5)

,Age,Overall,Potential,Value,Wage,Special,Preferred Foot,International Reputation,Weak Foot,Skill Moves,...,Contract_Years_Remaining,Contract_Info_Missing,Attack Work Rate,Defence Work Rate,Region_Africa,Region_Asia,Region_Europe,Region_North_America,Region_Oceania,Region_South_America
0,31,94,94,110500000.0,565000.0,2202,0,5.0,4.0,4.0,...,2.0,0,1,1,0.0,0.0,0.0,0.0,0.0,1.0
1,33,94,94,77000000.0,405000.0,2228,1,5.0,4.0,5.0,...,3.0,0,2,0,0.0,0.0,1.0,0.0,0.0,0.0
2,26,92,93,118500000.0,290000.0,2143,1,5.0,5.0,5.0,...,3.0,0,2,1,0.0,0.0,0.0,0.0,0.0,1.0
3,27,91,93,72000000.0,260000.0,1471,1,4.0,3.0,1.0,...,1.0,0,1,1,0.0,0.0,1.0,0.0,0.0,0.0
4,27,91,92,102000000.0,355000.0,2281,1,4.0,5.0,4.0,...,4.0,0,2,2,0.0,0.0,1.0,0.0,0.0,0.0


In [73]:
processed_df = processed_df.astype(float)
processed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18147 entries, 0 to 18146
Data columns (total 87 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       18147 non-null  float64
 1   Overall                   18147 non-null  float64
 2   Potential                 18147 non-null  float64
 3   Value                     18147 non-null  float64
 4   Wage                      18147 non-null  float64
 5   Special                   18147 non-null  float64
 6   Preferred Foot            18147 non-null  float64
 7   International Reputation  18147 non-null  float64
 8   Weak Foot                 18147 non-null  float64
 9   Skill Moves               18147 non-null  float64
 10  Body Type                 18147 non-null  float64
 11  Real Face                 18147 non-null  float64
 12  Jersey Number             18147 non-null  float64
 13  Height                    18147 non-null  float64
 14  Weight

## 8️⃣ Phase 8: Advanced Imputation (Machine Learning)
**Objective:** Accurately estimate missing financial data without distorting the distribution.
* **Problem:** ~8% of players were missing `Release Clause` data.
* **Solution:** Trained a **Random Forest Regressor** using high-correlation features (`Overall`, `Value`, `Wage`, `Age`).
* **Result:** Predicted missing release clauses with higher accuracy than simple mean/median imputation.

In [74]:
train_df = processed_df[processed_df['Release Clause'].notna()]
predict_df = processed_df[processed_df['Release Clause'].isna()]
X = train_df.drop(columns=['Release Clause'])
y = np.log1p(train_df['Release Clause'])

In [75]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import cross_val_score
# import numpy as np



# rf = RandomForestRegressor(
#     n_estimators=500,
#     max_depth=25,
#     min_samples_leaf=5,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X, y)


# scores = cross_val_score(
#     rf, X, y, cv=5, scoring='r2'
# )

# scores.mean()


In [76]:
# import joblib
# joblib.dump(rf, 'D:/desktop/MY_DOC/Projects/Comprehensive Analysis FIFA 19/models/release_clause_model.joblib')


In [77]:
import joblib
rf = joblib.load('D:/desktop/MY_DOC/Projects/Comprehensive Analysis FIFA 19/models/release_clause_model.joblib')



In [78]:
X_missing = predict_df.drop(columns=['Release Clause'])
pred_log = rf.predict(X_missing)

pred_release_clause = np.expm1(pred_log)

pred_release_clause_rounded = (
    (pred_release_clause / 1_000_000)
    .round(1)
    * 1_000_000
)

In [79]:
pred_release_clause_rounded

array([1.44e+08, 1.17e+08, 6.89e+07, ..., 1.00e+05, 1.00e+05, 0.00e+00],
      shape=(1275,))

In [80]:
df_display.loc[df_display['Release Clause'].isna(), 'Release Clause'] = pred_release_clause_rounded


In [81]:
predicted_df = df_display.loc[
    df_display.index.isin(predict_df.index),
    ['Release Clause']
]
predicted_df.head(10)


,Release Clause
28,144000000.0
38,117000000.0
91,68900000.0
166,66700000.0
176,61100000.0
332,40900000.0
354,35600000.0
357,40300000.0
427,37700000.0
434,32100000.0


In [82]:
len(df_display[df_display['Release Clause'].isna()])

0

## 9️⃣ Phase 9: Final Review & Export
**Objective:** Finalize the dataset structure by restoring readable labels and organizing columns for external visualization tools.

* **Process:**
    1. **Label Formatting:** Mapped numeric codes back to human-readable text to ensure dashboard tooltips are clear.
    2. **Column Reordering:** Organized the dataframe columns logically (Identity $\rightarrow$ Physical $\rightarrow$ Skills $\rightarrow$ Financials).
* **Export:** Saved `fifa19_eda_ready.csv` as the "Golden Record" for generating plots.
* **Next Step:** Proceed to `02_FIFA_Strategic_Analysis.ipynb` for visualization and build Tableau dashboards.

In [83]:
df_display.head(5)

,ID,Name,Age,Overall,Potential,Club,Value,Wage,Special,Preferred Foot,...,GKKicking,GKPositioning,GKReflexes,Release Clause,Years_At_Club,Contract_Years_Remaining,Contract_Info_Missing,Attack Work Rate,Defence Work Rate,Nationality_Region
0,158023,L. Messi,31,94,94,FC Barcelona,110500000.0,565000.0,2202,0,...,15.0,14.0,8.0,226500000.0,15.0,2.0,0,1,1,South_America
1,20801,Cristiano Ronaldo,33,94,94,Juventus,77000000.0,405000.0,2228,1,...,15.0,14.0,11.0,127100000.0,1.0,3.0,0,2,0,Europe
2,190871,Neymar Jr,26,92,93,Paris Saint-Germain,118500000.0,290000.0,2143,1,...,15.0,15.0,11.0,228100000.0,2.0,3.0,0,2,1,South_America
3,193080,De Gea,27,91,93,Manchester United,72000000.0,260000.0,1471,1,...,87.0,88.0,94.0,138600000.0,8.0,1.0,0,1,1,Europe
4,192985,K. De Bruyne,27,91,92,Manchester City,102000000.0,355000.0,2281,1,...,5.0,10.0,13.0,196400000.0,4.0,4.0,0,2,2,Europe


In [84]:
df_display['Preferred Foot'] = df_display['Preferred Foot'].map({
    0: 'Left',
    1: 'Right'
})

df_display['Body Type'] = df_display['Body Type'].map({
    0: 'Lean',
    1: 'Normal',
    2: 'Stocky'
})

df_display['Real Face'] = df_display['Real Face'].map({
    0: 'No',
    1: 'Yes'
})

reverse_W_Dec = {0: 'Low', 1: 'Medium', 2: 'High'}
df_display['Attack Work Rate'] = df_display['Attack Work Rate'].map(reverse_W_Dec)
df_display['Defence Work Rate'] = df_display['Defence Work Rate'].map(reverse_W_Dec)




In [85]:
ordered_columns = [
    # 1) Identifiers & display
    'ID', 'Name', 'Nationality_Region', 'Club', 'Position',

    # 2) Basic profile
    'Age', 'Height', 'Weight', 'Preferred Foot', 'Body Type', 'Real Face',

    # 3) Contract & value
    'Overall', 'Potential', 'International Reputation',
    'Value', 'Wage', 'Release Clause',
    'Contract_Years_Remaining', 'Years_At_Club', 'Contract_Info_Missing',

    # 4) Meta / mental
    'Special', 'Weak Foot', 'Skill Moves', 'Jersey Number',
    'Attack Work Rate', 'Defence Work Rate',

    # 5) Position ratings
    'LS','ST','RS','LW','LF','CF','RF','RW',
    'LAM','CAM','RAM','LM','LCM','CM','RCM','RM',
    'LWB','LDM','CDM','RDM','RWB',
    'LB','LCB','CB','RCB','RB',

    # 6) Technical attacking
    'Crossing','Finishing','HeadingAccuracy','ShortPassing','Volleys',
    'Dribbling','Curve','FKAccuracy','LongPassing','BallControl',

    # 7) Physical & movement
    'Acceleration','SprintSpeed','Agility','Reactions','Balance',
    'ShotPower','Jumping','Stamina','Strength','LongShots',

    # 8) Defensive skills
    'Aggression','Interceptions','Positioning','Vision','Penalties',
    'Composure','Marking','StandingTackle','SlidingTackle',

    # 9) Goalkeeper
    'GKDiving','GKHandling','GKKicking','GKPositioning','GKReflexes'
]

df_display = df_display[ordered_columns]
df_display.head(5)

,ID,Name,Nationality_Region,Club,Position,Age,Height,Weight,Preferred Foot,Body Type,...,Penalties,Composure,Marking,StandingTackle,SlidingTackle,GKDiving,GKHandling,GKKicking,GKPositioning,GKReflexes
0,158023,L. Messi,South_America,FC Barcelona,RF,31,170.18,72.12,Left,Stocky,...,75.0,96.0,33.0,28.0,26.0,6.0,11.0,15.0,14.0,8.0
1,20801,Cristiano Ronaldo,Europe,Juventus,ST,33,187.96,83.01,Right,Lean,...,85.0,95.0,28.0,31.0,23.0,7.0,11.0,15.0,14.0,11.0
2,190871,Neymar Jr,South_America,Paris Saint-Germain,LW,26,175.26,68.04,Right,Lean,...,81.0,94.0,27.0,24.0,33.0,9.0,9.0,15.0,15.0,11.0
3,193080,De Gea,Europe,Manchester United,GK,27,193.04,76.20,Right,Lean,...,40.0,68.0,15.0,21.0,13.0,90.0,85.0,87.0,88.0,94.0
4,192985,K. De Bruyne,Europe,Manchester City,RCM,27,180.34,69.85,Right,Normal,...,79.0,88.0,68.0,58.0,51.0,15.0,13.0,5.0,10.0,13.0


In [86]:
df_display.to_csv('D:/desktop/MY_DOC/Projects/Comprehensive Analysis FIFA 19/RawData/fifa19_eda_ready.csv', index=False)
